# Fine-Tune an LLM on AWS Trainium with LoRA

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong> This Immersion Day has been tested using the following SageMaker Distribution images:

<ul>
<li><strong>SageMaker Distribution Image 4.2</strong></li>
</ul>  
and the following SageMaker Python SDK version
<ul>
    <li><strong>SageMaker Python SDK version 3.13.1+</strong></li>
</ul>
</div>

## Overview

In this notebook you will:

1. Reference the pre-prepared training dataset in S3
2. Launch a LoRA fine-tuning job on **AWS Trainium** (`ml.trn1.2xlarge`) using the Hugging Face Optimum Neuron DLC
3. Produce a fine-tuned LoRA adapter saved to S3

**Model:** Configured in [`config.py`](config.py) — defaults to [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B). Both Qwen3-0.6B and Qwen3-1.7B are Apache 2.0 licensed, no gating.

**Why Trainium?** AWS Trainium chips are purpose-built for deep learning training. Combined with the [Neuron SDK](https://aws.amazon.com/machine-learning/neuron/) and [Optimum Neuron](https://huggingface.co/docs/optimum-neuron/index), you get cost-efficient fine-tuning with a familiar HuggingFace workflow.

**Why LoRA?** [Low-Rank Adaptation](https://arxiv.org/abs/2106.09685) trains only ~1% of model parameters, dramatically reducing memory and training time while preserving model quality.

**Estimated time:** ~5 minutes (with pre-compiled Neuron graphs from `00_workshop_prep.ipynb`)

**Notebook kernel:** Python 3 (ipykernel) on `ml.t3.large`

---
## 1. Setup

In [ ]:
# --- Lab dependencies (managed via uv) ---------------------------------------
# Installs THIS lab's complete, self-contained kernel dependencies from the
# lab requirements.txt using uv. Idempotent and fast when already satisfied.
# This is the only dependency step the lab needs - Setup.ipynb is not required.
import sys
!pip install -q uv
!uv pip install -q --python {sys.executable} -r requirements.txt

In [ ]:
import os
import json
import boto3
import sagemaker
from config import MODEL_ID
from sagemaker.core.helper.session_helper import Session, get_execution_role
from importlib.metadata import version

boto_session = boto3.Session()
sagemaker_session = Session(boto_session)
role = get_execution_role()

bucket = sagemaker_session.default_bucket()
region = boto_session.region_name

print(f"SageMaker SDK version: {version('sagemaker')}")
print(f"Role: {role}")
print(f"Bucket: {bucket}")
print(f"Region: {region}")

In [ ]:
# Training configuration
MAX_SEQ_LENGTH = 512
S3_PREFIX = "lab8-trainium-inferentia"

print(f"Model: {MODEL_ID} (configured in config.py)")
print(f"Sequence length: {MAX_SEQ_LENGTH}")
print(f"Region: {region}")
print(f"S3 prefix: {S3_PREFIX}")

---
## 2. Dataset (Pre-Prepared)

The training dataset has been pre-tokenized and uploaded to S3 by the workshop organizer
(see ). We just reference the S3 paths here.

In [ ]:
# S3 paths to pre-prepared dataset (uploaded by 00_workshop_prep.ipynb)
train_s3_uri = f"s3://{bucket}/{S3_PREFIX}/datasets/train"

print(f"Train data: {train_s3_uri}")


---
## 3. Launch Fine-Tuning on Trainium

We use the **HuggingFace Optimum Neuron DLC** which comes with:
- Neuron SDK 2.26
- PyTorch 2.8
- `optimum-neuron` 0.4.1 (upgraded to 0.4.2 at runtime for NeuronSFTTrainer)
- `transformers` 4.55.4 (upgraded to 4.57.x by optimum-neuron 0.4.2)
- `peft` 0.17.0, `trl` 0.24.0 (installed with optimum-neuron[training])

The training script (`src/train.py`) uses:
- `NeuronModelForCausalLM` — Neuron-native model loader for correct bf16 handling
- `NeuronSFTTrainer` — Supervised Fine-Tuning trainer optimized for Neuron hardware
- `LoraConfig` from PEFT — trains ~1% of parameters (all linear layers)
- Pre-compiled Neuron graphs loaded from S3 cache (no compilation needed)

The Neuron compile cache was prepared by `00_workshop_prep.ipynb` and is loaded
automatically via the `NEURON_COMPILE_CACHE_URL` environment variable.

In [ ]:
# Let's inspect the training script
!pygmentize src/train.py

In [ ]:
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import Compute, SourceCode, InputData
from sagemaker.core.shapes import OutputDataConfig

# HuggingFace Optimum Neuron DLC (Neuron SDK 2.26, PyTorch 2.8, optimum-neuron 0.4.1)
TRAINING_IMAGE = f"763104351884.dkr.ecr.{region}.amazonaws.com/huggingface-pytorch-training-neuronx:2.8.0-transformers4.55.4-neuronx-py310-sdk2.26.0-ubuntu22.04"

trainer = ModelTrainer(
    training_image=TRAINING_IMAGE,
    source_code=SourceCode(
        source_dir="src",
        entry_script="run_training.py",
    ),
    compute=Compute(
        instance_type="ml.trn1.2xlarge",
        instance_count=1,
        volume_size_in_gb=256,
    ),
    role=role,
    base_job_name="qwen25-lora-trn1",
    output_data_config=OutputDataConfig(s3_output_path=f"s3://{bucket}/{S3_PREFIX}/output"),
    hyperparameters={
        "model_id": MODEL_ID,
        "epochs": 1,
        "batch_size": 2,
        "learning_rate": 5e-5,
        "max_seq_length": MAX_SEQ_LENGTH,
        "lora_r": 16,
        "lora_alpha": 16,
        "gradient_accumulation_steps": 4,
    },
    environment={
        "MALLOC_ARENA_MAX": "64",
        "NEURON_FUSE_SOFTMAX": "1",
        "NEURON_CC_FLAGS": "--model-type=transformer --distribution-strategy=llm-training",
        "NEURON_COMPILE_CACHE_URL": f"s3://{bucket}/{S3_PREFIX}/neuron-cache/{MODEL_ID.split('/')[-1]}",
    },
)

print(f"Training image: {TRAINING_IMAGE.split('/')[-1]}")
print(f"Instance: ml.trn1.2xlarge (1 Trainium chip, 2 NeuronCores, 32GB HBM)")
print(f"Model: {MODEL_ID} (from config.py)")

In [ ]:
# Launch training!
trainer.train(
    input_data_config=[
        InputData(channel_name="train", data_source=train_s3_uri),
    ]
)

print("\n" + "=" * 60)
print("Training complete!")
print("=" * 60)

[07/14/26 20:20:25] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=5896783;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=5896784;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#110\110]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/14/26 20:20:26] INFO     Creating training_job resource.                                     ]8;id=5896789;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5896790;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31116\31116]8;;\

/opt/conda/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

[07/14/26 20:25:40] INFO     qwen25-lora-trn1-20260714202025/algo-1-1784060477:                  ]8;id=5896796;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5896797;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31439\31439]8;;\
                             Starting training script                                                              

                    INFO     qwen25-lora-trn1-20260714202025/algo-1-1784060477:                  ]8;id=5896802;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5896803;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31439\31439]8;;\
                             ++ /usr/local/bin/python3 --version                                                   

...

                    INFO     qwen25-lora-trn1-20260714202025/algo-1-1784060477:                  ]8;id=5897846;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5897847;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31439\31439]8;;\
                             with torch.cuda.amp.autocast(enabled=False):                                          

[07/14/26 20:26:31] INFO     qwen25-lora-trn1-20260714202025/algo-1-1784060477:                  ]8;id=5897852;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5897853;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31439\31439]8;;\
                             neuronxcc-2.21.18209.0+043b1bf7/MODULE_832253508781820108+3e47a5c8/                   
                             model.done not found in aws-neuron/optimum-neuron-cache: the                          
                             corresponding graph will be recompiled. This may take up to one                       
                             hour for large models.                                                                

                    INFO     qwen25-lora-trn1-20260714202025/algo-1-1784060477:                  ]8;id=5897858;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5897859;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31439\31439]8;;\
                             neuronxcc-2.21.18209.0+043b1bf7/MODULE_832253508781820108+3e47a5c8/                   
                             model.done not found in aws-neuron/optimum-neuron-cache: the                          
                             corresponding graph will be recompiled. This may take up to one                       
                             hour for large models.                                                                

                    INFO     qwen25-lora-trn1-20260714202025/algo-1-1784060477:                  ]8;id=5897864;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5897865;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31439\31439]8;;\
                             neuronxcc-2.21.18209.0+043b1bf7/MODULE_832253508781820108+3e47a5c8/                   
                             model.log not found in aws-neuron/optimum-neuron-cache: the                           
                             corresponding graph will be recompiled. This may take up to one                       
                             hour for large models.                                                                

                    INFO     qwen25-lora-trn1-20260714202025/algo-1-1784060477:                  ]8;id=5897870;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5897871;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31439\31439]8;;\
                             2026-07-14 20:26:23.000277:  37  INFO ||NEURON_CC_WRAPPER||: Call                     
                             compiler with cmd: neuronx-cc compile --framework=XLA                                 
                             /tmp/no-user/neuroncc_compile_workdir/debd4a79-4d15-4c33-8e0a-f7346                   
                             841529b/model.MODULE_832253508781820108+3e47a5c8.hlo_module.pb                        
                             --output                                                                              
                             /tmp/no-user/neuroncc_compile_workdir/debd4a79-4d15-4c33-8e0a-f7346                   
                             841529b/model.MODULE_832253508781820108+3e47a5c8.neff --target=trn1                   
                             --model-type=transformer --distribution-strategy=llm-training                         
                             --verbose=35                                                                          

In [ ]:
# Save training job name for the deployment notebook
training_job_name = trainer._latest_training_job.training_job_name
model_s3_uri = f"s3://{bucket}/{S3_PREFIX}/output/{training_job_name}/output/model.tar.gz"

with open("training_job_name.txt", "w") as f:
    f.write(training_job_name)

print(f"Training job name: {training_job_name}")
print(f"Model artifact:    {model_s3_uri}")
print(f"\nSaved to training_job_name.txt — the deployment notebook will pick this up automatically.")

---
## 4. What Just Happened?

Behind the scenes, the SageMaker training job:

1. **Launched** a `ml.trn1.2xlarge` instance with the HuggingFace Neuron DLC
2. **Installed** `optimum-neuron[training]==0.4.2` for NeuronSFTTrainer support
3. **Downloaded** the Qwen3-0.6B model from HuggingFace Hub
4. **Loaded** pre-compiled Neuron graphs from S3 cache (no compilation wait!)
5. **Trained** with LoRA using NeuronSFTTrainer — ~1% of parameters updated
6. **Saved** the LoRA adapter to S3

The pre-compiled cache eliminated the ~45 minute compilation step that would
otherwise be needed on the first run. This is why `00_workshop_prep.ipynb`
must be run once before this notebook.

---

## Next: Deploy to Inferentia2

Open [02_deployment.ipynb](02_deployment.ipynb) to deploy this model to a real-time endpoint on AWS Inferentia2.